## CODEX PDAC — StarDist prediction visualization

Counterpart of `CODEX_hcc/Pred_statistic_visual_hcc_all.ipynb`.

1. Load `Clinical_info` for annotated PDAC cores (278).
2. Plot **pooled StarDist ROC** for L2 / L12 / L1.
3. Compare per-core **macro AUROC** across **coverslip** and **SAMPLE_LABEL**.

Shared helpers: `Hist2Pheno_pkg/uni_label_cv_helpers.py`. Clinical loaders: `CODEX_pdac/s1167_plot.py`.


In [ ]:
## Setup (CODEX PDAC)
import matplotlib
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

import sys
import warnings
import logging
from pathlib import Path

warnings.filterwarnings("ignore")
logging.getLogger("ome_zarr").setLevel(logging.ERROR)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
CODE_DIR = REPO / "code"
PKG_DIR = CODE_DIR / "Hist2Pheno_pkg"
PDAC_CODE_DIR = CODE_DIR / "CODEX_pdac"

for _p in (PKG_DIR, PDAC_CODE_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

PAN_ORGAN = "codex_pdac"
DATA_ROOT = REPO / "data" / "CODEX" / "HCC" / "Michael_data_transfer" / "s1167"
STARDIST_RESULT_ROOT = DATA_ROOT / "result_all_spatial" / "stardist"
OUT_DIR = DATA_ROOT / "result_all_spatial" / "clinical_viz_pdac"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("OUT_DIR  :", OUT_DIR)
print("pan_organ:", PAN_ORGAN)


## Load clinical information

`raw_metadata_updated.xlsx` sheet `Clinical_info`, `cohort='PDAC'`, annotated cores only.
Demo ACQ: `Charvill-94_c001_v001_r001_reg001`.


In [ ]:
import importlib
import s1167_plot
importlib.reload(s1167_plot)
from s1167_plot import load_s1167_clinical_info

clinical = load_s1167_clinical_info(cohort="PDAC", annotated_only=True)


## Pooled StarDist ROC (annotated PDAC cores)

Symlink the cores present in `clinical`, then plot pooled OvR ROC for `l2` / `l12` / `l1`.


In [ ]:
import importlib
import uni_label_cv_helpers as uni_nb
import s1167_plot

importlib.reload(uni_nb)
importlib.reload(s1167_plot)
from s1167_plot import PDAC_STARDIST_MACRO_AUROC_TIERS
from uni_label_cv_helpers import plot_pooled_stardist_tier_rocs, symlink_stardist_sample_dirs

filtered_root = symlink_stardist_sample_dirs(
    STARDIST_RESULT_ROOT,
    clinical["ACQUISITION_ID"],
    OUT_DIR / "_tmp_selected_stardist",
)

pooled_roc = plot_pooled_stardist_tier_rocs(
    filtered_root,
    OUT_DIR,
    PDAC_STARDIST_MACRO_AUROC_TIERS,
    layout="pooled_stardist",
    pan_organ=PAN_ORGAN,
    figsize={"l12": (2.0, 2.0), "l1": (2.0, 2.0)},
)


## StarDist macro AUROC vs clinical groups

Per-core macro AUROC across **coverslip** and **SAMPLE_LABEL**.

- `method="rank"`: Mann–Whitney U (2 groups) or Kruskal–Wallis (3+)
- `method="parametric"`: Welch t-test / one-way ANOVA


In [ ]:
import importlib
import s1167_plot
importlib.reload(s1167_plot)
from s1167_plot import analyze_pdac_stardist_macro_auroc_by_clinical

stardist_auroc_clinical = analyze_pdac_stardist_macro_auroc_by_clinical(
    filtered_root,
    clinical,
    method="rank",
    pan_organ=PAN_ORGAN,
)
